# Session 1: Research Questions, Data & Measurement
### Case Study: What Does the Human Development Index Measure?
*Bayesian Cognitive Science & Data Analysis (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/01_research_questions_data_measurement.ipynb)

---

## 1. Setup & Environment Initialization
This cell automatically detects whether you are running on Google Colab or locally. If running in Colab, it downloads the course dataset directly from GitHub.


In [ ]:
# ==============================================================================
# 🚀 1. Setup Cell: Auto-Detect Environment & Fetch Data
# ==============================================================================
import sys
import os
import urllib.request
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Running in Google Colab environment.")
    import plotly.io as pio
    pio.renderers.default = "colab"
else:
    print("💻 Running in local environment.")

# Auto-fetch dataset from GitHub if not already present
DATA_DIR = "data"
DATA_FILE = "hdi_state.csv"
LOCAL_PATH = os.path.join(DATA_DIR, DATA_FILE)
RAW_URL = f"https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/{DATA_FILE}"

if not os.path.exists(LOCAL_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"📥 Downloading {DATA_FILE} from GitHub...")
    urllib.request.urlretrieve(RAW_URL, LOCAL_PATH)
    print(f"✅ Successfully downloaded to {LOCAL_PATH}")
else:
    print(f"📂 Found existing local dataset at {LOCAL_PATH}")

df = pd.read_csv(LOCAL_PATH)
print(f"📊 Dataset loaded: {len(df)} rows, columns: {list(df.columns)}")
df.head()


## 2. Inspecting the Composite Measurement
We filter the 50 states with observed personal income in 2000 and calculate:
1. State income rank (1 = highest income, 50 = lowest income).
2. HDI rank (1 = highest HDI, 50 = lowest HDI).
3. The rank gain $\Delta = \text{Rank}_{\text{Income}} - \text{Rank}_{\text{HDI}}$ (positive values mean the state achieves better human development than income rank alone predicts).


In [ ]:
# Prepare 50 states (excluding DC which lacks survey income in this historical table)
valid = df.dropna(subset=["income_2000"]).copy()
valid["rank_income"] = valid["income_2000"].rank(ascending=False)
valid["rank_hdi_50"] = valid["hdi"].rank(ascending=False)
valid["rank_gain"] = valid["rank_income"] - valid["rank_hdi_50"]

r_pearson, p_pearson = pearsonr(valid["income_2000"], valid["hdi"])
r_spearman, p_spearman = spearmanr(valid["rank_income"], valid["rank_hdi_50"])

print(f"Pearson Correlation (Raw Scale):   r = {r_pearson:.3f} (p = {p_pearson:.2e})")
print(f"Spearman Correlation (Rank Scale): r_s = {r_spearman:.3f} (p = {p_spearman:.2e})")


## 3. Investigating Measurement Discrepancies
### The Kentucky vs. South Carolina Puzzle
Gelman, Hill, and Vehtari (*Regression and Other Stories*, Ch. 2) noted that South Carolina and Kentucky have virtually identical average personal incomes, yet diverge dramatically on HDI. Let us inspect their exact values:


In [ ]:
sc_ky = valid[valid["state_abb"].isin(["SC", "KY"])][["state", "income_2000", "rank_income", "hdi", "rank_hdi_50"]]
print(sc_ky.to_string(index=False))

diff_income = abs(sc_ky.iloc[0]["income_2000"] - sc_ky.iloc[1]["income_2000"])
diff_hdi = abs(sc_ky.iloc[0]["hdi"] - sc_ky.iloc[1]["hdi"])
print(f"\nIncome difference: ${diff_income:.2f}")
print(f"HDI difference:    {diff_hdi:.3f} (spanning {diff_hdi / 0.163 * 100:.1f}% of national HDI range!)")


## 4. Interactive Rank-Rank Calibration Plot


In [ ]:
fig = go.Figure()

# Scatter of states
fig.add_trace(go.Scatter(
    x=valid["rank_income"],
    y=valid["rank_hdi_50"],
    mode="markers+text",
    text=valid["state_abb"],
    textposition="top center",
    marker=dict(
        size=9,
        color=valid["rank_gain"],
        colorscale="Viridis",
        colorbar=dict(title="Rank Gain"),
        showscale=True
    ),
    hovertemplate="<b>%{text}</b><br>Income Rank: #%{x:.0f}<br>HDI Rank: #%{y:.0f}<extra></extra>"
))

# 45-degree diagonal line
fig.add_trace(go.Scatter(
    x=[1, 50], y=[1, 50],
    mode="lines",
    line=dict(color="red", dash="dash"),
    name="Equal Rank (y = x)"
))

fig.update_layout(
    title=f"Income Rank vs. HDI Rank Across 50 U.S. States (r_s = {r_spearman:.2f})",
    xaxis_title="Income Rank (1 = Highest)",
    yaxis_title="HDI Rank (1 = Highest)",
    template="plotly_white",
    height=500
)

fig.show()
